In [1]:
import pandas as pd
from slider import config

table = pd.read_csv(config.RUNS_TABLE)
print(len(table), "runs")
table.head()

718 runs


,file,session,dataset,wave,players,topology,success,end,duration,bot,T,agree_error
0,log_Fri Sep 16 161000 EDT 2016.txt,lab,CL,1,7,Ring,False,failed,300.0,False,2056,0.003333
1,log_Fri Sep 16 161000 EDT 2016.txt,lab,CL,2,7,Hierarchy,True,success,59.0,False,470,0.003333
2,log_Fri Sep 16 161000 EDT 2016.txt,lab,CL,3,7,Full,True,success,90.0,False,734,0.004286
3,log_Fri Sep 16 161000 EDT 2016.txt,lab,CL,4,7,Hierarchy,True,success,80.0,False,730,0.003333
4,log_Fri Sep 16 161000 EDT 2016.txt,lab,CL,5,7,Full,True,success,21.0,False,156,0.004286


In [2]:
cl = table[(table["dataset"] == "CL") & (table["success"] == True)]

stats = cl.groupby("topology")["duration"].agg(["count", "mean", "std"])
stats = stats.loc[["Hierarchy", "Full"]]
stats.columns = ["n", "mean (s)", "s.d. (s)"]

stats.round(2)

,n,mean (s),s.d. (s)
topology,,,
Hierarchy,58,61.61,52.97
Full,52,87.01,62.04


In [3]:
# 2.2 fraught_mask(run): True for each row that is a fraught state (in src/slider/fraught.py)
import numpy as np
from slider import slider_logs as sl
from slider.fraught import fraught_mask, fraught_percent, fraught_percent_interp

runs = sl.parse_all()   # same order as the rows of runs_table.csv
assert all(run["file"] == f and run["wave"] == w for run, f, w in zip(runs, table["file"], table["wave"]))

# a run with no logged rows gets an empty mask
masks = [fraught_mask(run) if len(run["t"]) > 0 else np.zeros(0, dtype=bool) for run in runs]

In [4]:
# sanity checks
has_rows = [i for i in range(len(runs)) if len(masks[i]) > 0]
print("(a) runs with rows:", len(has_rows),
      "| first row fraught:", sum(masks[i][0] for i in has_rows))

won = [i for i in has_rows if table["success"][i]]
print("(b) successful runs:", len(won),
      "| last row fraught:", sum(masks[i][-1] for i in won))

full = table.index[(table["dataset"] == "CL") & (table["topology"] == "Full")]
print("(c) CL Full runs:", len(full),
      "| rows:", sum(len(masks[i]) for i in full),
      "| fraught rows:", sum(masks[i].sum() for i in full))

hier = table.index[(table["dataset"] == "CL") & (table["topology"] == "Hierarchy")]
hier_won = [i for i in hier if table["success"][i]]
print("(d) CL Hierarchy runs:", len(hier),
      "| with at least one fraught row:", sum(masks[i].any() for i in hier))
print("    successful only:", len(hier_won),
      "| with at least one fraught row:", sum(masks[i].any() for i in hier_won))

(a) runs with rows: 710 | first row fraught: 0
(b) successful runs: 609 | last row fraught: 0
(c) CL Full runs: 63 | rows: 43240 | fraught rows: 0
(d) CL Hierarchy runs: 63 | with at least one fraught row: 30
    successful only: 58 | with at least one fraught row: 27


In [5]:
# 2.3 % of each round spent fraught (forward hold: each row lasts until the next row)
# hier = the CL Hierarchy rounds, from the cell above
h = table.loc[hier, ["file", "wave", "success", "duration"]].copy()
h["fraught % (time)"] = [fraught_percent(i, runs, masks, table)[0] for i in hier]
h["fraught % (rows)"] = [fraught_percent(i, runs, masks, table)[1] for i in hier]

print("CL Hierarchy rounds:", len(h))
print("mean % fraught, time-weighted:", round(h["fraught % (time)"].mean(), 2))
print("mean % fraught, row-counted:  ", round(h["fraught % (rows)"].mean(), 2))

CL Hierarchy rounds: 63
mean % fraught, time-weighted: 11.58
mean % fraught, row-counted:   9.18


In [6]:
# 2.4 substantially fraught: more than 10% of the round (time-weighted)
big = h[h["fraught % (time)"] > 10]
print("rounds with > 10% fraught:", len(big))
print("of those, succeeded:", big["success"].sum())
big.round(2)

rounds with > 10% fraught: 18
of those, succeeded: 15


,file,wave,success,duration,fraught % (time),fraught % (rows)
1,log_Fri Sep 16 161000 EDT 2016.txt,2,True,59.00,18.64,19.36
3,log_Fri Sep 16 161000 EDT 2016.txt,4,True,80.00,11.25,10.14
10,log_Fri Sep 16 165753 EDT 2016.txt,2,True,55.02,10.36,7.92
19,log_Mon Sep 19 164446 EDT 2016.txt,2,False,297.82,58.26,41.72
37,log_Tue Sep 20 142130 EDT 2016.txt,2,True,67.74,12.55,12.91
39,log_Tue Sep 20 142130 EDT 2016.txt,4,True,55.30,45.39,36.24
56,log_Wed Sep 14 170705 EDT 2016.txt,3,True,213.00,86.85,71.79
61,log_Wed Sep 21 160602 EDT 2016.txt,1,False,300.08,19.63,18.75
125,log_v09_Fri Oct 07 162937 EDT 2016.txt,2,True,50.06,17.78,11.38
261,log_v09_Mon Sep 26 173549 EDT 2016.txt,7,True,158.16,40.71,32.52


In [7]:
# 2.4 footnote: 18 vs. 19.
# The authors' Matlab code (Code/runapem.m, "Time warp") does not hold each row until the next one.
# It joins the rows with straight lines, takes 1001 evenly spaced times from the first row to the
# last row, and counts how many of those states are fraught. Same thing here (fraught_percent_interp in src/slider/fraught.py):
h["fraught % (interp)"] = [fraught_percent_interp(i, runs) for i in hier]

big_interp = h[h["fraught % (interp)"] > 10]
print("mean % fraught, interpolated:", round(h["fraught % (interp)"].mean(), 2))
print("rounds with > 10% fraught:", len(big_interp))
print("of those, succeeded:", big_interp["success"].sum())

# the round that is above 10% here but not with the forward hold
big_interp[~big_interp.index.isin(big.index)].round(2)

mean % fraught, interpolated: 12.23
rounds with > 10% fraught: 19
of those, succeeded: 16


,file,wave,success,duration,fraught % (time),fraught % (rows),fraught % (interp)
482,log_v09_Wed Sep 28 163724 EDT 2016.txt,9,True,36.84,2.99,0.74,25.87


In [8]:
# data caveat: some early lab logs only have whole-second timestamps
rows = []
for f in table["file"].unique():
    idx = table.index[table["file"] == f]
    t = np.concatenate([runs[i]["t"] for i in idx])
    if len(t) > 0:
        rows.append([f, table["dataset"][idx[0]], len(t), 100 * np.mean(t != np.round(t))])

coarse = pd.DataFrame(rows, columns=["file", "dataset", "rows", "% non-integer t"])
print("files:", len(coarse), "| files with whole-second timestamps only:", (coarse["% non-integer t"] == 0).sum())
print("smallest % non-integer t among the other files:", round(coarse[coarse["% non-integer t"] > 0]["% non-integer t"].min(), 1))

coarse_files = coarse[coarse["% non-integer t"] == 0]["file"]
in_coarse = table["file"].isin(coarse_files)
print("CL runs in those files:", (in_coarse & (table["dataset"] == "CL")).sum(), "of", (table["dataset"] == "CL").sum())
print("CL Hierarchy rounds in those files:", in_coarse[hier].sum(), "of", len(hier))
coarse[coarse["% non-integer t"] == 0]

files: 83 | files with whole-second timestamps only: 4
smallest % non-integer t among the other files: 94.7
CL runs in those files: 25 of 187
CL Hierarchy rounds in those files: 9 of 63


,file,dataset,rows,% non-integer t
0,log_Fri Sep 16 161000 EDT 2016.txt,CL,7458,0.0
5,log_Wed Sep 14 155221 EDT 2016.txt,CL,5036,0.0
6,log_Wed Sep 14 170705 EDT 2016.txt,CL,6479,0.0
57,log_Wed Sep 14 180549 EDT 2016.txt,NaN,6664,0.0
